In [6]:
import cariaco_obs

In [7]:
cariaco_obs.load_cariaco_targets()

(array([0.096978  , 0.08511578, 0.19976537, 0.05937712, 0.03302277,
        1.46705197, 0.43246982, 3.182666  ]),
 ['Pico (<2 µm)',
  'Nano (2-20 µm)',
  'Micro (>20 µm)',
  'Zoo >200 µm',
  'Zoo >500 µm',
  'NO3',
  'PP',
  'Export'],
 [{'label': 'Pico (<2 µm)',
   'column': 'pico_mmolN',
   'type': 'phyto',
   'size_min': 0.0,
   'size_max': 2.0},
  {'label': 'Nano (2-20 µm)',
   'column': 'nano_mmolN',
   'type': 'phyto',
   'size_min': 2.0,
   'size_max': 20.0},
  {'label': 'Micro (>20 µm)',
   'column': 'micro_mmolN',
   'type': 'phyto',
   'size_min': 20.0,
   'size_max': inf},
  {'label': 'Zoo >200 µm',
   'column': 'zoo_gt200_mmolN',
   'type': 'zoo',
   'size_min': 200.0,
   'size_max': inf},
  {'label': 'Zoo >500 µm',
   'column': 'zoo_gt500_mmolN',
   'type': 'zoo',
   'size_min': 500.0,
   'size_max': inf},
  {'label': 'NO3', 'column': 'NO3_mmolN', 'type': 'nutrient'},
  {'label': 'PP', 'column': 'PP_mmolN_m3_d', 'type': 'pp'},
  {'label': 'Export',
   'column': 'export_flu

In [8]:
import numpy as np
import pandas as pd
import cariaco_obs

# Raw CSV — sees columns the loader strips
df = pd.read_csv(cariaco_obs.DEFAULT_CSV_PATH)
df["_date"] = pd.to_datetime(df["date"])

print(f"=== Shape ===")
print(f"  rows: {len(df)}   cols: {df.shape[1]}")

print(f"\n=== All columns (non-NA count, % filled) ===")
counts = df.notna().sum().sort_values(ascending=False)
for c, n in counts.items():
    if c.startswith("_"):  # skip helper cols
        continue
    print(f"  {c:35s} {n:4d}  ({100*n/len(df):5.1f}%)")

# --- Phyto size-class coverage ---
phyto_cols = ["pico_mmolN", "nano_mmolN", "micro_mmolN"]
present_phyto = [c for c in phyto_cols if c in df.columns]
print(f"\n=== Phyto size-class coverage ===")
for c in present_phyto:
    print(f"  {c:18s} {df[c].notna().sum()} months")
if len(present_phyto) == 3:
    all_three = df[phyto_cols].notna().all(axis=1).sum()
    any_one   = df[phyto_cols].notna().any(axis=1).sum()
    print(f"  any one present:    {any_one} months")
    print(f"  all three present:  {all_three} months")

# --- Derived size-spectrum metrics (if they live in the CSV) ---
metric_cols = ["size_centroid", "size_shannon", "nbss_slope"]
print(f"\n=== Derived size-spectrum metrics ===")
for c in metric_cols:
    if c in df.columns:
        s = df[c].dropna()
        if len(s):
            print(f"  {c:14s} n={len(s):3d}  mean={s.mean():+.3f}  "
                  f"median={s.median():+.3f}  IQR=[{s.quantile(0.25):+.3f}, "
                  f"{s.quantile(0.75):+.3f}]  min={s.min():+.3f}  max={s.max():+.3f}")
        else:
            print(f"  {c:14s} n=0 (all NaN)")
    else:
        print(f"  {c:14s} NOT IN CSV")

# --- Intersected coverage: each scatter panel = points per pair ---
env_cols = ["FN_mmolN_m2_d", "NO3_mmolN", "PP_mmolN_m3_d", "TotChlA_mmolN",
            "Chl_niskin_mmolN", "PhaeoChl_ratio",
            "zoo_gt200_mmolN", "zoo_gt500_mmolN",
            "Temp_C", "sst", "Isotherm_21", "depth_cutoff"]
y_for_intersect = "size_centroid" if "size_centroid" in df.columns else (
    "micro_mmolN" if "micro_mmolN" in df.columns else None)
if y_for_intersect:
    print(f"\n=== Pairwise non-NA coverage: {y_for_intersect} × env ===")
    for c in env_cols:
        if c in df.columns:
            n = (df[y_for_intersect].notna() & df[c].notna()).sum()
            print(f"  {y_for_intersect} × {c:22s} n={n}")
        else:
            print(f"  {y_for_intersect} × {c:22s} COL MISSING")

# --- Coverage by regime (binary upwelling) ---
def _row(sub):
    n_any = sub[phyto_cols].notna().any(axis=1).sum()  if len(present_phyto)==3 else None
    n_all = sub[phyto_cols].notna().all(axis=1).sum()  if len(present_phyto)==3 else None
    n_cen = sub["size_centroid"].notna().sum() if "size_centroid" in sub.columns else None
    return n_any, n_all, n_cen

if "upwelling" in df.columns:
    print(f"\n=== Coverage by upwelling regime ===")
    print(f"  {'regime':14s} {'rows':>5s} {'any_phyto':>10s} {'all_phyto':>10s} {'centroid':>9s}")
    for reg, sub in df.groupby("upwelling", dropna=False):
        n_any, n_all, n_cen = _row(sub)
        print(f"  {str(reg):14s} {len(sub):5d} {n_any!s:>10s} {n_all!s:>10s} {n_cen!s:>9s}")

if "ui" in df.columns:
    print(f"\n=== Coverage by ui (4-level) ===")
    print(f"  {'ui':14s} {'rows':>5s} {'any_phyto':>10s} {'all_phyto':>10s} {'centroid':>9s}")
    for reg, sub in df.groupby("ui", dropna=False):
        n_any, n_all, n_cen = _row(sub)
        print(f"  {str(reg):14s} {len(sub):5d} {n_any!s:>10s} {n_all!s:>10s} {n_cen!s:>9s}")

# --- Era split (pre/post sardine collapse cutoff = 2005-01-01) ---
print(f"\n=== Coverage by era (cutoff 2005-01-01) ===")
print(f"  {'era':14s} {'rows':>5s} {'any_phyto':>10s} {'all_phyto':>10s} {'centroid':>9s}")
for era, sub in [("pre-2005", df[df["_date"] < "2005-01-01"]),
                 ("post-2005", df[df["_date"] >= "2005-01-01"])]:
    n_any, n_all, n_cen = _row(sub)
    print(f"  {era:14s} {len(sub):5d} {n_any!s:>10s} {n_all!s:>10s} {n_cen!s:>9s}")

=== Shape ===
  rows: 255   cols: 43

=== All columns (non-NA count, % filled) ===
  date                                 255  (100.0%)
  cutoff_source                        255  (100.0%)
  time_month                           255  (100.0%)
  year                                 255  (100.0%)
  month                                255  (100.0%)
  sst                                  229  ( 89.8%)
  MLD                                  229  ( 89.8%)
  upwelling                            228  ( 89.4%)
  Isotherm_21                          228  ( 89.4%)
  temp_50m                             228  ( 89.4%)
  ui                                   228  ( 89.4%)
  PhaeoChl_ratio                       198  ( 77.6%)
  Chl_niskin_mgm3                      198  ( 77.6%)
  Phaeo_niskin_mgm3                    198  ( 77.6%)
  Chl_niskin_mmolN                     198  ( 77.6%)
  n_niskin_samples                     198  ( 77.6%)
  depth_cutoff                         198  ( 77.6%)
  Temp_C        